In [1]:
# Day 2: Data Cleaning
# Goal: Clean the messy raw data

import pandas as pd
from datetime import datetime

# Load raw data
print("Loading raw data...")
df = pd.read_csv('customer_complaints_raw.csv')
print(f"Original dataset: {len(df)} rows\n")

print("="*60)
print("CLEANING STEP 1: REMOVE DUPLICATE COMPLAINT IDs")
print("="*60)

# Count duplicates before
duplicates_before = df.duplicated(subset=['complaint_id']).sum()
print(f"Duplicates found: {duplicates_before}")

# Keep first occurrence, remove rest
df = df.drop_duplicates(subset=['complaint_id'], keep='first')
print(f"After removing duplicates: {len(df)} rows")
print(f"Removed: {duplicates_before} duplicate rows\n")

print("="*60)
print("CLEANING STEP 2: FIX DATE FORMATS")
print("="*60)

# Show some examples before cleaning
print("Sample dates BEFORE cleaning:")
print(df['date_submitted'].head(10).tolist())

# Convert all date formats to standard YYYY-MM-DD
def clean_date(date_str):
    """Convert various date formats to YYYY-MM-DD"""
    if pd.isna(date_str):
        return None
    
    # Try different date formats
    formats = ['%Y-%m-%d', '%m/%d/%y', '%B %d, %Y', '%d-%m-%Y']
    
    for fmt in formats:
        try:
            date_obj = datetime.strptime(str(date_str), fmt)
            return date_obj.strftime('%Y-%m-%d')
        except:
            continue
    
    return None

df['date_submitted'] = df['date_submitted'].apply(clean_date)

print("\nSample dates AFTER cleaning:")
print(df['date_submitted'].head(10).tolist())
print()

print("="*60)
print("CLEANING STEP 3: STANDARDIZE TEXT COLUMNS")
print("="*60)

# Columns to clean (lowercase and trim spaces)
text_columns = ['channel', 'product_category', 'product_name', 
                'store_location', 'priority_level', 'repeat_complaint']

for col in text_columns:
    print(f"\nCleaning: {col}")
    print(f"  Before: {df[col].unique()[:5]}")
    
    # Convert to string, lowercase, remove extra spaces
    df[col] = df[col].astype(str).str.lower().str.strip()
    
    # Replace 'nan' string with actual NaN
    df[col] = df[col].replace('nan', None)
    
    print(f"  After:  {df[col].unique()[:5]}")

print()

print("="*60)
print("CLEANING STEP 4: FIX PRODUCT NAMES (Standardize)")
print("="*60)

# Standardize common product name variations
product_mapping = {
    'iphone14': 'iphone 14',
    'iphone 14': 'iphone 14',
    'macbook pro': 'macbook pro',
    'airpods': 'airpods pro',
    'airpods pro': 'airpods pro'
}

print("Sample products BEFORE standardization:")
print(df['product_name'].value_counts().head(10))

df['product_name'] = df['product_name'].replace(product_mapping)

print("\nSample products AFTER standardization:")
print(df['product_name'].value_counts().head(10))
print()

print("="*60)
print("CLEANING STEP 5: FIX OUTLIERS IN RESOLUTION TIME")
print("="*60)

print(f"Resolution time statistics BEFORE cleaning:")
print(df['resolution_time_hours'].describe())

# Remove impossible values (negative, 0, or extremely high)
print(f"\nRows with resolution_time <= 0: {(df['resolution_time_hours'] <= 0).sum()}")
print(f"Rows with resolution_time > 500: {(df['resolution_time_hours'] > 500).sum()}")

# Set outliers to NaN (we'll handle later)
df.loc[df['resolution_time_hours'] <= 0, 'resolution_time_hours'] = None
df.loc[df['resolution_time_hours'] > 500, 'resolution_time_hours'] = None

print(f"\nResolution time statistics AFTER cleaning:")
print(df['resolution_time_hours'].describe())
print()

print("="*60)
print("CLEANING STEP 6: HANDLE MISSING VALUES")
print("="*60)

# Count missing values
missing_before = df.isnull().sum()
print("Missing values in each column:")
print(missing_before[missing_before > 0])

# Fill missing priority levels based on keywords in complaint text
print("\nFilling missing priority levels...")

def assign_priority(row):
    """Assign priority based on complaint keywords"""
    if pd.notna(row['priority_level']) and row['priority_level'] != '':
        return row['priority_level']
    
    text = str(row['complaint_text']).lower()
    
    # Critical keywords
    if any(word in text for word in ['broken', 'defective', 'not working']):
        return 'high'
    # Medium keywords
    elif any(word in text for word in ['delayed', 'wrong', 'missing']):
        return 'medium'
    # Default
    else:
        return 'low'

df['priority_level'] = df.apply(assign_priority, axis=1)

# Fill missing store locations for non-store channels
df.loc[df['channel'].isin(['email', 'phone', 'chat', 'social media']), 'store_location'] = 'online'

# Fill empty product categories with 'other'
df['product_category'] = df['product_category'].replace('', 'other')
df['product_category'] = df['product_category'].fillna('other')

# Fill empty product names with 'unknown'
df['product_name'] = df['product_name'].replace('', 'unknown')
df['product_name'] = df['product_name'].fillna('unknown')

# For repeat_complaint, assume 'no' if missing
df['repeat_complaint'] = df['repeat_complaint'].replace('', 'no')
df['repeat_complaint'] = df['repeat_complaint'].fillna('no')

print("\nMissing values AFTER cleaning:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])
print()

print("="*60)
print("CLEANING STEP 7: CLEAN COMPLAINT TEXT")
print("="*60)

# Basic text cleaning
df['complaint_text'] = df['complaint_text'].str.lower()
df['complaint_text'] = df['complaint_text'].str.strip()

print("Sample complaint text:")
print(df['complaint_text'].head(3).tolist())
print()

print("="*60)
print("CLEANING COMPLETE!")
print("="*60)

print(f"\nFinal dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"\nData types:")
print(df.dtypes)

# Save cleaned data
df.to_csv('customer_complaints_cleaned.csv', index=False)
print("\n✅ Cleaned data saved as: customer_complaints_cleaned.csv")

print("\n" + "="*60)
print("SUMMARY OF CHANGES")
print("="*60)
print(f"1. Removed {duplicates_before} duplicate rows")
print(f"2. Standardized all dates to YYYY-MM-DD format")
print(f"3. Converted all text to lowercase and trimmed spaces")
print(f"4. Standardized product names (combined variations)")
print(f"5. Fixed {(df['resolution_time_hours'].isna()).sum()} outlier resolution times")
print(f"6. Filled missing priority levels using complaint keywords")
print(f"7. Cleaned complaint text")


Loading raw data...
Original dataset: 25525 rows

CLEANING STEP 1: REMOVE DUPLICATE COMPLAINT IDs
Duplicates found: 525
After removing duplicates: 25000 rows
Removed: 525 duplicate rows

CLEANING STEP 2: FIX DATE FORMATS
Sample dates BEFORE cleaning:
['2022-09-23', '11-12-2023', 'May 19, 2023', '12-09-2022', '05/05/24', '09/13/23', '03/04/24', 'June 04, 2022', '28-04-2023', '27-02-2023']

Sample dates AFTER cleaning:
['2022-09-23', '2023-12-11', '2023-05-19', '2022-09-12', '2024-05-05', '2023-09-13', '2024-03-04', '2022-06-04', '2023-04-28', '2023-02-27']

CLEANING STEP 3: STANDARDIZE TEXT COLUMNS

Cleaning: channel
  Before: ['In-Store' 'Social Media' 'Email' 'email' 'PHONE']
  After:  ['in-store' 'social media' 'email' 'phone' 'chat']

Cleaning: product_category
  Before: ['Phones' 'Smart Home' 'laptop' nan 'Other']
  After:  ['phones' 'smart home' 'laptop' None 'other']

Cleaning: product_name
  Before: ['MacBook Pro' 'Ring Doorbell' 'airpods' 'Nest Thermostat' 'HP Laptop']
  After: